# Dự báo nguy cơ ngập theo tuyến đường

**Sổ Nghiên Cứu AI, dự án mẫu 2/10.** Kết hợp lượng mưa, mực triều và lịch sử ngập để cảnh báo trước vài giờ.

**Câu hỏi nghiên cứu:** Thêm dữ liệu mực triều vào mô hình chỉ dùng lượng mưa có cải thiện khả năng dự báo ngập không?

**Giả thuyết:** Mô hình mưa cộng triều đạt recall cao hơn mô hình chỉ dùng mưa ít nhất 10 điểm phần trăm với cùng mức báo động giả.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

Lượng mưa thật được tải từ Open-Meteo (miễn phí, không cần tài khoản). **Mực triều và nhãn ngập trong bản mẫu là giả lập** để quy trình chạy được; em phải thay bằng dữ liệu tự thu thập.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)

import requests
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import recall_score, precision_score, roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.inspection import permutation_importance

## 1. Lượng mưa theo giờ

In [ ]:
LAT, LON = 21.03, 105.85          # đổi thành tọa độ nơi em nghiên cứu
BAT_DAU, KET_THUC = "2023-01-01", "2024-12-31"
def tai_mua():
    url = ("https://archive-api.open-meteo.com/v1/archive?latitude=%s&longitude=%s&start_date=%s&end_date=%s"
           "&hourly=precipitation&timezone=auto") % (LAT, LON, BAT_DAU, KET_THUC)
    j = requests.get(url, timeout=60).json()
    return pd.DataFrame({"thoi_gian": pd.to_datetime(j["hourly"]["time"]), "mua": j["hourly"]["precipitation"]})
try:
    mua = tai_mua(); NGUON_MUA = "Open-Meteo"
except Exception as e:
    print("Không tải được Open-Meteo (", type(e).__name__, "), dùng lượng mưa giả lập.")
    t = pd.date_range(BAT_DAU, KET_THUC + " 23:00", freq="h")
    mua_mua = np.isin(t.month, [5, 6, 7, 8, 9, 10])
    mua = pd.DataFrame({"thoi_gian": t, "mua": np.where(np.random.rand(len(t)) < np.where(mua_mua, .12, .03), np.random.gamma(.8, 6, len(t)), 0)})
    NGUON_MUA = "GIẢ LẬP"
mua["mua"] = mua["mua"].fillna(0)
print(NGUON_MUA, "|", len(mua), "giờ | tổng mưa", round(mua.mua.sum()), "mm")
mua.groupby(mua.thoi_gian.dt.to_period("M")).mua.sum().plot(kind="bar", figsize=(11, 3), title="Tổng lượng mưa theo tháng (mm)"); plt.show()

## 2. Mực triều và nhãn ngập
Nếu em có file `trieu.csv` (cột `thoi_gian, muc_trieu`) và `diem_ngap.csv` (cột `thoi_gian, tuyen_duong, ngap` với `ngap` là 0 hoặc 1), đặt vào Colab và chạy lại. Nếu chưa có, ô dưới tạo dữ liệu giả lập **chỉ để luyện quy trình**.

In [ ]:
if os.path.exists("trieu.csv"):
    trieu = pd.read_csv("trieu.csv", parse_dates=["thoi_gian"])
else:
    h = (mua.thoi_gian - mua.thoi_gian.min()).dt.total_seconds() / 3600
    trieu = pd.DataFrame({"thoi_gian": mua.thoi_gian,
        "muc_trieu": 1.0 * np.sin(2 * np.pi * h / 12.42) * (0.7 + 0.3 * np.cos(2 * np.pi * h / 354)) + 0.25 * np.sin(2 * np.pi * h / 24.84)})
    print("Mực triều GIẢ LẬP (dao động bán nhật triều)")
df = mua.merge(trieu, on="thoi_gian", how="left").sort_values("thoi_gian").reset_index(drop=True)
df["muc_trieu"] = df["muc_trieu"].interpolate().bfill()
for k in [1, 3, 6, 24]: df[f"mua_{k}h"] = df["mua"].rolling(k, min_periods=1).sum()
df["gio"], df["thang"] = df.thoi_gian.dt.hour, df.thoi_gian.dt.month

TUYEN = {"Đường A (trũng)": 0.0, "Đường B": 0.6, "Đường C (cao)": 1.2}     # độ cao tương đối, em tự khảo sát
if os.path.exists("diem_ngap.csv"):
    nhan = pd.read_csv("diem_ngap.csv", parse_dates=["thoi_gian"]); NGUON_NHAN = "của em"
else:
    rows = []
    for ten, cao in TUYEN.items():
        z = 0.12 * df["mua_3h"] + 0.03 * df["mua_24h"] + 1.4 * df["muc_trieu"] - 1.5 * cao - 2.8 + np.random.normal(0, .5, len(df))
        rows.append(pd.DataFrame({"thoi_gian": df.thoi_gian, "tuyen_duong": ten, "ngap": (z > 0).astype(int)}))
    nhan = pd.concat(rows); NGUON_NHAN = "GIẢ LẬP"
print("Nhãn ngập:", NGUON_NHAN, "| tỉ lệ giờ ngập:", round(nhan.ngap.mean() * 100, 2), "%")

## 3. Bảng đặc trưng, dự báo trước 3 giờ
Đặc trưng tại giờ *t*, nhãn là có ngập ở giờ *t + 3* hay không. Như vậy mô hình thật sự **dự báo**, không chỉ mô tả.

In [ ]:
TRUOC = 3
# Triều có bảng dự báo công bố trước, nên được phép dùng mực triều DỰ BÁO tại giờ t + 3. Lượng mưa tương lai thì không.
df["trieu_du_bao"] = df["muc_trieu"].shift(-TRUOC)
X = []
for ten, cao in TUYEN.items():
    n = nhan[nhan.tuyen_duong == ten][["thoi_gian", "ngap"]].copy()
    n["thoi_gian"] = n["thoi_gian"] - pd.Timedelta(hours=TRUOC)
    d = df.merge(n, on="thoi_gian"); d["do_cao"] = cao; d["tuyen"] = ten; X.append(d)
D = pd.concat(X).dropna()
DT_MUA = ["mua", "mua_1h", "mua_3h", "mua_6h", "mua_24h", "do_cao", "gio", "thang"]
DT_DU = DT_MUA + ["muc_trieu", "trieu_du_bao"]
# Chia theo THỜI GIAN: học năm đầu, kiểm tra năm sau. Không trộn ngẫu nhiên vì các giờ liền nhau rất giống nhau.
moc = D.thoi_gian.min() + (D.thoi_gian.max() - D.thoi_gian.min()) * 0.5
train, test = D[D.thoi_gian < moc], D[D.thoi_gian >= moc]
print(f"Huấn luyện: {len(train)} dòng | Kiểm tra: {len(test)} dòng | Tỉ lệ ngập kiểm tra {test.ngap.mean():.3%}")

## 4. So sánh mô hình

In [ ]:
MH = {
 "Logistic, chỉ mưa (đối chứng)": (LogisticRegression(max_iter=2000, class_weight="balanced"), DT_MUA),
 "Logistic, mưa + triều": (LogisticRegression(max_iter=2000, class_weight="balanced"), DT_DU),
 "Rừng ngẫu nhiên, mưa + triều": (RandomForestClassifier(300, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=SEED), DT_DU),
 "Gradient boosting, mưa + triều": (HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED), DT_DU),
}
KQ, PROB = {}, {}
for ten, (m, cols) in MH.items():
    m.fit(train[cols], train.ngap); p = m.predict_proba(test[cols])[:, 1]; PROB[ten] = p
    # Chọn ngưỡng trên tập HUẤN LUYỆN sao cho chỉ 5% giờ không ngập bị báo động giả, rồi áp dụng cho tập kiểm tra
    p_tr = m.predict_proba(train[cols])[:, 1]; nguong = np.quantile(p_tr[train.ngap.values == 0], 0.95)
    yhat = (p >= nguong).astype(int)
    KQ[ten] = {"Recall (báo giả 5%)": recall_score(test.ngap, yhat), "Precision": precision_score(test.ngap, yhat, zero_division=0),
               "AUC": roc_auc_score(test.ngap, p), "AP (PR-AUC)": average_precision_score(test.ngap, p)}
print(pd.DataFrame(KQ).T.round(3))
plt.figure(figsize=(6, 4.5))
for ten, p in PROB.items():
    pr, rc, _ = precision_recall_curve(test.ngap, p); plt.plot(rc, pr, label=ten)
plt.xlabel("Recall (bắt được bao nhiêu giờ ngập)"); plt.ylabel("Precision"); plt.legend(fontsize=7); plt.title("Đánh đổi recall và precision"); plt.show()

## 5. Yếu tố nào quyết định? (tầm quan trọng hoán vị)

In [ ]:
m, cols = MH["Gradient boosting, mưa + triều"]
imp = permutation_importance(m, test[cols], test.ngap, scoring="average_precision", n_repeats=5, random_state=SEED)
pd.Series(imp.importances_mean, index=cols).sort_values().plot(kind="barh", figsize=(6, 3.5), title="Giảm AP khi xáo trộn từng đặc trưng"); plt.show()

## Việc của em
1. **Nhãn thật:** lập `diem_ngap.csv` từ báo chí địa phương, ảnh, hỏi người dân. Mỗi dòng ghi nguồn trong sổ.
2. **Triều thật:** lấy số liệu mực nước, triều do đài khí tượng thủy văn khu vực công bố (nếu địa phương em chịu ảnh hưởng triều). Nếu không có triều, thay bằng mực nước sông hoặc bỏ đặc trưng này.
3. **Đo cảnh báo sớm:** thử `TRUOC = 1, 3, 6` giờ, vẽ recall theo số giờ báo trước.
4. **Ngưỡng:** chọn ngưỡng xác suất sao cho báo động giả không quá 1 lần mỗi tuần, ghi lý do.